# 🎓 Predicting Student Performance with a Data Science Approach

**Based on:** [Medium Article by Jorge Bernal](https://medium.com/@jtrejo_91009/predicting-student-performance-with-a-data-science-approach-c8fe1d66057d)

**Dataset:** UCI Machine Learning Repository — Student Performance Dataset  
**Goal:** Analyze factors that influence final grades (G3) and build a predictive Linear Regression model.

---
### Pipeline Overview
1. Download Dataset (auto, no manual upload needed)
2. Import Libraries
3. Load & Explore Data
4. General Data Analysis
5. Correlation Analysis
6. Feature Engineering
7. Build Linear Regression Model
8. Evaluate the Model
9. Conclusion

---
## STEP 1 — Download the Dataset
The dataset is downloaded directly from the UCI repository — **no manual upload required** in Google Colab.

In [ ]:
import zipfile
import urllib.request
import os

# Download the zip archive from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
zip_path = "student.zip"

print("Downloading dataset from UCI...")
urllib.request.urlretrieve(url, zip_path)

# Extract both CSV files
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(".")

# Confirm files are present
files = [f for f in os.listdir('.') if f.startswith('student')]
print("✅ Extracted files:", files)

---
## STEP 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

print("✅ All libraries imported successfully!")

---
## STEP 3 — Load & Explore the Data

The article uses two datasets:
- `student-mat.csv` — Math course
- `student-por.csv` — Portuguese language course

Both use **semicolon (`;`)** as separator. We combine them for a unified analysis.

In [ ]:
# Load both datasets (semicolon-separated)
df_mat = pd.read_csv('student-mat.csv', sep=';')
df_por = pd.read_csv('student-por.csv', sep=';')

print("Math dataset shape    :", df_mat.shape)
print("Portuguese dataset shape:", df_por.shape)

print("\n--- Math Dataset (first 5 rows) ---")
df_mat.head()

In [ ]:
print("--- Portuguese Dataset (first 5 rows) ---")
df_por.head()

In [ ]:
# Add a source label and combine into one DataFrame
df_mat['source'] = 'math'
df_por['source'] = 'portuguese'

df_students = pd.concat([df_mat, df_por], ignore_index=True)

print("Combined dataset shape:", df_students.shape)
print("Columns:", list(df_students.columns))

---
## STEP 4 — General Data Analysis
Check for missing values and plot the distribution of all features.

In [ ]:
# Check for missing values
missing = df_students.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found!")

In [ ]:
# Basic statistics for numeric columns
df_students.describe()

In [ ]:
# Plot distributions of all numeric features
numeric_df = df_students.select_dtypes(include=['number'])

numeric_df.hist(bins=15, figsize=(20, 15), color='steelblue', edgecolor='white')
plt.suptitle('Feature Distributions', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom in: distribution of the target variable G3 (final grade)
plt.figure(figsize=(8, 5))
sns.histplot(df_students['G3'], bins=20, kde=True, color='steelblue')
plt.title('Distribution of Final Grades (G3)', fontsize=14)
plt.xlabel('Final Grade (G3)')
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"G3 mean  : {df_students['G3'].mean():.2f}")
print(f"G3 median: {df_students['G3'].median():.2f}")
print(f"G3 range : {df_students['G3'].min()} – {df_students['G3'].max()}")

---
## STEP 5 — Correlation Analysis
Compute and visualize the correlation matrix to understand which features most influence the final grade (G3).

In [ ]:
# Select only numeric columns for correlation
numeric_columns = df_students.select_dtypes(include=['number'])

# Compute correlation matrix
correlation_matrix = numeric_columns.corr()

# Plot full heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Correlation Matrix — All Numeric Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Focus: correlation of each feature with G3 (final grade)
g3_corr = correlation_matrix['G3'].drop('G3').sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in g3_corr]
g3_corr.plot(kind='bar', color=colors, edgecolor='white')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Final Grade (G3)', fontsize=14)
plt.ylabel('Pearson Correlation')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\nTop 5 positively correlated features with G3:")
print(g3_corr.head())
print("\nTop 5 negatively correlated features with G3:")
print(g3_corr.tail())

---
## STEP 6 — Feature Engineering
Convert categorical variables into numeric dummy variables so they can be used in the regression model.

In [ ]:
# Drop the helper 'source' column before encoding
df_model = df_students.drop(columns=['source'])

# Convert categorical variables into dummy/indicator variables
df_students_encoded = pd.get_dummies(df_model, drop_first=True)

print("Shape before encoding:", df_model.shape)
print("Shape after encoding :", df_students_encoded.shape)
print("\nAll columns after encoding:")
print(list(df_students_encoded.columns))

In [ ]:
# Preview the encoded dataset
df_students_encoded.head()

---
## STEP 7 — Build the Linear Regression Model
Split data into training (80%) and testing (20%) sets, then fit a Linear Regression model to predict G3.

In [ ]:
# Define features (X) and target variable (y)
X = df_students_encoded.drop('G3', axis=1)
y = df_students_encoded['G3']

print("Features shape:", X.shape)
print("Target shape  :", y.shape)

In [ ]:
# Split into training and testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

In [ ]:
# Fit the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

print("✅ Model trained successfully!")
print("\nSample predictions vs actual:")
comparison = pd.DataFrame({'Actual': y_test.values[:10], 'Predicted': y_pred[:10].round(2)})
print(comparison.to_string(index=False))

---
## STEP 8 — Evaluate the Model
Use Mean Squared Error (MSE) and R² score to assess model performance.

In [ ]:
# Calculate evaluation metrics
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("=" * 35)
print("  Model Evaluation Results")
print("=" * 35)
print(f"  Mean Squared Error  (MSE) : {mse:.4f}")
print(f"  Root MSE           (RMSE) : {rmse:.4f}")
print(f"  R² Score                  : {r2:.4f}")
print("=" * 35)

In [ ]:
# Plot: Actual vs Predicted grades
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='k', linewidths=0.3, s=60)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.title('Actual vs Predicted Final Grades (G3)', fontsize=14)
plt.xlabel('Actual G3')
plt.ylabel('Predicted G3')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Plot: Residuals (errors)
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=25, kde=True, color='coral')
plt.axvline(0, color='black', linestyle='--', linewidth=1.2)
plt.title('Residuals Distribution (Actual - Predicted)', fontsize=14)
plt.xlabel('Residual')
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 most influential features (by absolute coefficient value)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).reindex(columns=['Feature', 'Coefficient'])

coef_df['Abs_Coef'] = coef_df['Coefficient'].abs()
top15 = coef_df.sort_values('Abs_Coef', ascending=False).head(15)

plt.figure(figsize=(10, 6))
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in top15['Coefficient']]
plt.barh(top15['Feature'], top15['Coefficient'], color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 15 Features by Regression Coefficient', fontsize=14)
plt.xlabel('Coefficient Value')
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

---
## STEP 9 — Conclusion

### Key Findings

| Finding | Detail |
|---|---|
| **Most influential features** | G1 and G2 (prior period grades) have the highest positive correlation with G3 |
| **Parental education** | Medu (mother) and Fedu (father) show a moderate positive effect |
| **Study time** | Positively associated with better final grades |
| **Failures** | Strong negative correlation — past failures predict lower G3 |
| **Alcohol consumption** | Dalc (daily) and Walc (weekend) show weak negative correlation |

### Model Performance
- A low MSE and moderate-to-high R² indicate the Linear Regression model explains a good portion of variance in final grades.
- The residuals are roughly centered at zero, suggesting no major systematic bias.

### Possible Next Steps
- Try **Random Forest** or **Gradient Boosting** for better accuracy
- Apply **cross-validation** for more robust evaluation
- Perform **feature selection** to remove low-importance variables
- Analyze math and Portuguese datasets **separately** to compare patterns